# Notebook 02 — Stylometric Feature Extraction

Extracts 63 handcrafted stylometric features from every email across four categories using NLTK, SpaCy, and Textstat.

## What This Notebook Does
- Loads the preprocessed dataset from Notebook 01
- Defines four feature extraction functions:
  - Lexical features (20) — word length, vocabulary richness, punctuation
  - Phishing-specific features (20) — URLs, urgency keywords, threat language
  - Readability features (10) — Flesch-Kincaid, Gunning Fog, SMOG, Coleman-Liau
  - Syntactic/POS features (13) — noun/verb/adjective ratios via SpaCy
- Runs extraction on all emails with progress tracking
- Saves feature matrix to disk

## Inputs
- data/processed/dataset_phase1.csv (from Notebook 01)

## Outputs
- data/processed/stylometric_features.csv — 63 features, 4,000 emails
- data/processed/stylometric_features_final.csv — 63 features, 10,492 emails

## Runtime
Approximately 10-20 minutes depending on dataset size

## Dependencies
- NLTK (punkt, stopwords, averaged_perceptron_tagger)
- SpaCy (en_core_web_sm)
- Textstat

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import nltk
import spacy
import textstat
import re
import string
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Paths
BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Load dataset
df = pd.read_csv(DATA_PROCESSED / "dataset_phase1.csv")
print(f"Dataset loaded: {len(df)} emails")
print(df['label'].value_counts().sort_index())

In [ ]:
#basic text features
def extract_basic_features(text):
    """Extract basic lexical and structural features from email text."""
    text = str(text)
    words = text.split()
    sentences = nltk.sent_tokenize(text)
    
    # Avoid division by zero
    num_words = len(words) if len(words) > 0 else 1
    num_sentences = len(sentences) if len(sentences) > 0 else 1
    
    features = {
        # Length features
        'email_length': len(text),
        'num_words': len(words),
        'num_sentences': num_sentences,
        'num_paragraphs': len([p for p in text.split('\n\n') if p.strip()]),
        
        # Word features
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'max_word_length': max([len(w) for w in words]) if words else 0,
        'avg_sentence_length': num_words / num_sentences,
        
        # Vocabulary richness
        'unique_words': len(set(words)),
        'type_token_ratio': len(set(words)) / num_words,
        'vocab_richness': len(set(w.lower() for w in words)) / num_words,
        
        # Punctuation features
        'num_exclamations': text.count('!'),
        'num_questions': text.count('?'),
        'num_commas': text.count(','),
        'num_periods': text.count('.'),
        'exclamation_ratio': text.count('!') / num_words,
        'question_ratio': text.count('?') / num_words,
        
        # Capital letters
        'num_capitals': sum(1 for c in text if c.isupper()),
        'capital_ratio': sum(1 for c in text if c.isupper()) / len(text) if text else 0,
        
        # Special characters
        'num_special_chars': sum(1 for c in text if c in string.punctuation),
        'special_char_ratio': sum(1 for c in text if c in string.punctuation) / len(text) if text else 0,
    }
    return features

# Test on one email
sample = df['text'].iloc[0]
features = extract_basic_features(sample)
print(f"Features extracted: {len(features)}")
print("\nSample values:")
for k, v in list(features.items())[:8]:
    print(f"  {k}: {v:.4f}")

In [ ]:
#url and phishing specific features
def extract_phishing_features(text):
    """Extract features specifically useful for phishing detection."""
    text = str(text)
    words = text.split()
    num_words = len(words) if len(words) > 0 else 1
    text_lower = text.lower()
    
    # URL detection
    url_pattern = re.compile(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+')
    urls = url_pattern.findall(text)
    
    # Phishing keyword lists
    urgency_words = ['urgent', 'immediately', 'expire', 'suspend', 'verify', 
                     'confirm', 'update', 'click', 'login', 'password', 
                     'account', 'bank', 'limited', 'offer', 'winner', 'prize',
                     'free', 'congratulations', 'selected', 'act now']
    
    greeting_words = ['dear', 'hello', 'hi', 'greetings', 'good morning', 
                      'good afternoon', 'dear customer', 'dear user']
    
    threat_words = ['suspended', 'terminated', 'blocked', 'restricted', 
                    'unauthorized', 'illegal', 'fraud', 'risk']
    
    features = {
        # URL features
        'num_urls': len(urls),
        'has_url': int(len(urls) > 0),
        'url_ratio': len(urls) / num_words,
        'num_http': text_lower.count('http://'),
        'num_https': text_lower.count('https://'),
        
        # Phishing keywords
        'urgency_word_count': sum(1 for w in urgency_words if w in text_lower),
        'threat_word_count': sum(1 for w in threat_words if w in text_lower),
        'has_greeting': int(any(g in text_lower for g in greeting_words)),
        
        # Email structure signals
        'has_unsubscribe': int('unsubscribe' in text_lower),
        'has_dear': int('dear' in text_lower),
        'has_winner': int('winner' in text_lower or 'won' in text_lower),
        'has_free': int('free' in text_lower),
        'has_click_here': int('click here' in text_lower),
        'has_verify': int('verify' in text_lower or 'verification' in text_lower),
        'has_account': int('account' in text_lower),
        'has_password': int('password' in text_lower),
        'has_bank': int('bank' in text_lower),
        'has_invoice': int('invoice' in text_lower or 'payment' in text_lower),
        
        # Digit features
        'num_digits': sum(c.isdigit() for c in text),
        'digit_ratio': sum(c.isdigit() for c in text) / len(text) if text else 0,
    }
    return features

# Test on one email
features = extract_phishing_features(df['text'].iloc[0])
print(f"Features extracted: {len(features)}")
print("\nSample values:")
for k, v in list(features.items())[:8]:
    print(f"  {k}: {v:.4f}")

In [ ]:
#readability features
def extract_readability_features(text):
    """Extract readability scores - these differ between human and AI writing."""
    text = str(text)
    
    # Need at least some content to calculate scores
    if len(text.split()) < 10:
        return {
            'flesch_reading_ease': 0,
            'flesch_kincaid_grade': 0,
            'gunning_fog': 0,
            'smog_index': 0,
            'coleman_liau_index': 0,
            'automated_readability_index': 0,
            'dale_chall_readability': 0,
            'difficult_words': 0,
            'linsear_write_formula': 0,
            'text_standard': 0,
        }
    
    try:
        features = {
            # Readability scores
            # Higher flesch = easier to read (AI tends to score differently)
            'flesch_reading_ease': textstat.flesch_reading_ease(text),
            'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
            'gunning_fog': textstat.gunning_fog(text),
            'smog_index': textstat.smog_index(text),
            'coleman_liau_index': textstat.coleman_liau_index(text),
            'automated_readability_index': textstat.automated_readability_index(text),
            'dale_chall_readability': textstat.dale_chall_readability_score(text),
            'difficult_words': textstat.difficult_words(text),
            'linsear_write_formula': textstat.linsear_write_formula(text),
            # text_standard returns a string like "8th and 9th grade" - extract number
            'text_standard': float(str(textstat.text_standard(text, float_output=True))),
        }
    except Exception as e:
        print(f"Error: {e}")
        features = {k: 0 for k in ['flesch_reading_ease', 'flesch_kincaid_grade',
                                    'gunning_fog', 'smog_index', 'coleman_liau_index',
                                    'automated_readability_index', 'dale_chall_readability',
                                    'difficult_words', 'linsear_write_formula', 'text_standard']}
    return features

# Test on one email
features = extract_readability_features(df['text'].iloc[0])
print(f"Features extracted: {len(features)}")
print("\nSample values:")
for k, v in features.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
#pos and syntactic features
def extract_syntactic_features(text):
    """Extract part-of-speech and syntactic features using spaCy."""
    text = str(text)
    
    # Truncate very long emails to speed up spaCy processing
    # spaCy struggles with very long texts
    if len(text) > 5000:
        text = text[:5000]
    
    doc = nlp(text)
    
    total_tokens = len(doc) if len(doc) > 0 else 1
    
    # Count POS tags
    pos_counts = {}
    for token in doc:
        pos = token.pos_
        pos_counts[pos] = pos_counts.get(pos, 0) + 1
    
    features = {
        # POS ratios (normalised by total tokens)
        'noun_ratio': pos_counts.get('NOUN', 0) / total_tokens,
        'verb_ratio': pos_counts.get('VERB', 0) / total_tokens,
        'adj_ratio': pos_counts.get('ADJ', 0) / total_tokens,
        'adv_ratio': pos_counts.get('ADV', 0) / total_tokens,
        'pronoun_ratio': pos_counts.get('PRON', 0) / total_tokens,
        'propn_ratio': pos_counts.get('PROPN', 0) / total_tokens,
        'det_ratio': pos_counts.get('DET', 0) / total_tokens,
        'punct_ratio': pos_counts.get('PUNCT', 0) / total_tokens,
        'num_ratio': pos_counts.get('NUM', 0) / total_tokens,
        
        # Sentence structure
        'num_entities': len(doc.ents),
        'entity_ratio': len(doc.ents) / total_tokens,
        
        # Stopword ratio (AI text tends to have different stopword usage)
        'stopword_ratio': sum(1 for token in doc if token.is_stop) / total_tokens,
        
        # Punctuation diversity
        'unique_punct': len(set(token.text for token in doc if token.is_punct)),
    }
    return features

# Test on one email
features = extract_syntactic_features(df['text'].iloc[0])
print(f"Features extracted: {len(features)}")
print("\nSample values:")
for k, v in features.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
def extract_all_features(text):
    """Run all four feature extractors and combine into one dictionary."""
    features = {}
    features.update(extract_basic_features(text))
    features.update(extract_phishing_features(text))
    features.update(extract_readability_features(text))
    features.update(extract_syntactic_features(text))
    return features

# Test on one email
sample_features = extract_all_features(df['text'].iloc[0])
print(f"Total features: {len(sample_features)}")
print("\nAll feature names:")
for i, key in enumerate(sample_features.keys()):
    print(f"  {i+1:2d}. {key}")

In [ ]:
from tqdm import tqdm
tqdm.pandas()

print("Extracting features for all emails")

# Process each email
all_features = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    try:
        features = extract_all_features(row['text'])
        features['label'] = row['label']
        all_features.append(features)
    except Exception as e:
        # If one email fails, skip it and continue
        print(f"Skipped email {idx}: {e}")
        continue

# Convert to dataframe
features_df = pd.DataFrame(all_features)


print(f"Shape: {features_df.shape}")
print(f"Emails processed: {len(features_df)}")
print(f"Features per email: {features_df.shape[1] - 1}")  # minus label column
print(f"\nMissing values:\n{features_df.isnull().sum().sum()} total")

In [ ]:
save_path = DATA_PROCESSED / "stylometric_features.csv"
features_df.to_csv(save_path, index=False)

print(f"Saved to: {save_path}")
print(f"File size: {save_path.stat().st_size / (1024*1024):.2f} MB")
print(f"\nFirst 3 rows preview:")
print(features_df.head(3))

In [ ]:
#run this if needed next time
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("C:/phishing_detection")
DATA_PROCESSED = BASE_DIR / "data" / "processed"

# Load already-processed features
features_df = pd.read_csv(DATA_PROCESSED / "stylometric_features.csv")
print(f"Features loaded: {features_df.shape}")
print(f"Emails: {len(features_df)}, Features: {features_df.shape[1]-1}")